# Amaliy mashgʻulot 9. Nomlangan obyektlarni aniqlash: BIO-belgilash va CRF modeli

9-laboratoriyada tokenlarni bosib BIO-teglarni qoʻydingiz, belgilarni yoqib-oʻchirdingiz va CRF notoʻgʻri ketma-ketliklarni qanday tuzatishini koʻrdingiz. Bu yerda xuddi shuni kodda bajaramiz:

1. belgilangan matn boʻlaklarini BIO-teglarga va aksincha oʻtkazamiz, notoʻgʻri ketma-ketliklarni topishni oʻrganamiz;
2. har bir tokenni belgilar toʻplami bilan tasvirlaymiz: soʻz shakli, qoʻshnilar va oʻzbek tili uchun eng muhimi — kelishik qoʻshimchalari (-da, -dan, -ga, -ning) va bosh harf;
3. `sklearn-crfsuite` dagi CRF ni 45 ta kichik gapda oʻqitamiz va oʻrganilgan oʻtish vaznlarini koʻramiz;
4. CRF ni oʻzimiz Viterbi algoritmi bilan dekodlaymiz va $P(y \mid x)$ ni hisoblaymiz — model qanday tuzilganini tushunganimizga ishonch hosil qilamiz;
5. obyektlar darajasida aniqlik, toʻliqlik va F1 funksiyasini yozamiz va belgilar toʻplamlarini solishtiramiz.

*«NLP algoritmlari» kursining 9-laboratoriyasiga amaliy mashgʻulot. Kataklarni yuqoridan pastga ishga tushiring: Shift+Enter yoki «Runtime → Run all». Javoblaringizni saqlash uchun «File → Save a copy in Drive» ni bosing.*

In [ ]:
!pip install -q sklearn-crfsuite

## 1. BIO-belgilash

Nomlangan obyekt — bir yoki bir nechta soʻzdan iborat boʻlak: «Alisher Navoiy» (PER, shaxs), «Hirot» (LOC, joy), «Oʻzbekiston Milliy universiteti» (ORG, tashkilot). Boʻlaklarni qidirishni ketma-ketlikni belgilashga (8-amaliy mashgʻulotdagi kabi) keltirish uchun har bir tokenga teg beriladi: **B-X** — X turidagi obyektning birinchi tokeni, **I-X** — davomi, **O** — obyektdan tashqarida.

Quyidagi korpus shu amaliy mashgʻulot uchun yozilgan: 45 ta oʻqitish va 18 ta test gapi, obyektlar `[matn](TUR)` koʻrinishida ajratilgan. Birinchi gap — laboratoriyadagi misol.

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

TRAIN_TEXT = """
[Alisher Navoiy](PER) [Hirotda](LOC) tugʻilgan va [Samarqanddagi](LOC) madrasada oʻqigan
[Mirzo Ulugʻbek](PER) [Samarqandda](LOC) rasadxona qurdirgan
[Abdulla Qodiriy](PER) [Toshkentda](LOC) tugʻilgan
[Anvar](PER) [Toshkent davlat texnika universitetida](ORG) oʻqiydi
[Dilnoza](PER) [Buxorodan](LOC) [Toshkentga](LOC) koʻchib keldi
[Oʻzbekiston Milliy universiteti](ORG) yangi bino qurdi
[Paxtakor](ORG) jamoasi [Fargʻonada](LOC) oʻynadi
[Gʻafur Gʻulomning](PER) sheʼrlarini bolalar sevib oʻqiydi
[Nodira](PER) [Fanlar akademiyasida](ORG) ishlaydi
Yozda biz [Xivaga](LOC) bordik
[Sardor Karimov](PER) [Andijondan](LOC) keldi
[Oʻzbekiston havo yoʻllari](ORG) [Samarqandga](LOC) yangi reys ochdi
[Bobur](PER) [Andijonda](LOC) tugʻilgan
[Zahiriddin Muhammad Bobur](PER) [Hindistonda](LOC) saltanat qurgan
[Madina](PER) va [Jasur](PER) [Toshkent](LOC) koʻchalarida sayr qilishdi
[Amudaryo](LOC) [Orol dengiziga](LOC) quyiladi
[Sirdaryo](LOC) [Fargʻona vodiysi](LOC) orqali oqadi
[Ibn Sino](PER) [Buxoro](LOC) yaqinidagi qishloqda tugʻilgan
[Toshkent davlat texnika universiteti](ORG) talabalari tanlovda gʻolib boʻldi
[Aziz Rahimov](PER) [Oʻzbekiston Milliy bankida](ORG) ishlaydi
Bugun havo juda issiq boʻldi
[Navoiy](LOC) shahrida yangi zavod ochildi
[Navoiyning](PER) gʻazallari koʻp tillarga tarjima qilingan
[Kamola](PER) [Samarqanddan](LOC) qaytdi
[Shahzoda Yusupova](PER) [Oʻzbekiston Milliy universitetida](ORG) dars beradi
Yozda biz [Chorvoqqa](LOC) dam olishga bordik
[Rustam](PER) har kuni ertalab yuguradi
[Toshkent](LOC) [Oʻzbekistonning](LOC) poytaxti
[Paxtakor](ORG) [Toshkentda](LOC) gʻalaba qozondi
Kecha kutubxonada yangi kitoblar koʻrdim
[Jahongir](PER) [Qoʻqonga](LOC) ishga ketdi
[Fanlar akademiyasi](ORG) olimlari yangi kashfiyot qilishdi
[Laylo Hasanova](PER) [Namanganda](LOC) tugʻilgan
Bu yil [Buxoroda](LOC) xalqaro festival boʻladi
[Oybek](PER) va [Abdulla Qahhor](PER) mashhur yozuvchilar
[Ulugʻbek](PER) rasadxonasi [Samarqandda](LOC) joylashgan
[Bekzod](PER) [Toshkent davlat texnika universitetiga](ORG) hujjat topshirdi
[Oʻzbekiston havo yoʻllari](ORG) samolyoti [Toshkentga](LOC) qoʻndi
Bolalar hovlida toʻp oʻynashdi
[Farhod](PER) [Termizdan](LOC) [Qarshiga](LOC) poyezdda bordi
[Abdulla Qodiriyning](PER) romanlari maktabda oʻrganiladi
Kuzda [Fargʻona vodiysida](LOC) paxta terimi boshlanadi
[Malika](PER) [Andijondagi](LOC) maktabda ishlaydi
[Alisher Navoiy](PER) koʻp asarlarini [Hirotda](LOC) yozgan
[Sevara](PER) [Xivadan](LOC) chiroyli suratlar olib keldi
"""
TEST_TEXT = """
[Alisher Navoiy](PER) [Samarqandda](LOC) ham yashagan
[Dilnoza Karimova](PER) [Oʻzbekiston Milliy universitetiga](ORG) oʻqishga kirdi
[Anvar](PER) [Xivadan](LOC) [Buxoroga](LOC) bordi
[Sardor](PER) [Fanlar akademiyasiga](ORG) maqola yubordi
[Toshkentdagi](LOC) universitetlarda talabalar koʻp
[Navoiy](LOC) shahridan [Buxorogacha](LOC) avtobus qatnaydi
[Paxtakor](ORG) [Andijonda](LOC) oʻynaydi
[Jasur Rahimov](PER) [Qoʻqondan](LOC) keldi
[Bobur](PER) [Kobulni](LOC) egallagan
[Madina](PER) [Oʻzbekiston havo yoʻllarida](ORG) ishlaydi
[Ibn Sinoning](PER) kitoblari koʻp tillarga tarjima qilingan
Kecha [Samarqanddan](LOC) mehmonlar keldi
Ertalab havo salqin edi
[Nodira Hasanova](PER) [Termizda](LOC) tugʻilgan
[Kamola](PER) [Orol dengizi](LOC) haqida maqola yozdi
[Malika](PER) opa bozordan olma oldi
[Shahzod](PER) [Oʻzbektelekomga](ORG) dasturchi boʻlib ishga kirdi
[Dilshod](PER) [Nukusdan](LOC) [Urganchga](LOC) koʻchdi
"""

def parse_line(line):
    # qatorni tahlil qilish: [matn](TUR) — obyekt, qolganlari — alohida soʻzlar
    tokens, spans = [], []
    for m in re.finditer(r"\[([^\]]+)\]\((PER|LOC|ORG)\)|(\S+)", line):
        if m.group(3):
            tokens.append(m.group(3))
        else:
            words = m.group(1).split()
            spans.append((len(tokens), len(tokens) + len(words), m.group(2)))
            tokens += words
    return tokens, spans

def spans_to_bio(n, spans):
    # obyektning birinchi soʻzi — B-TUR, keyingilari — I-TUR, qolgan hammasi — O
    tags = ["O"] * n
    for start, end, typ in spans:
        tags[start] = "B-" + typ
        for i in range(start + 1, end):
            tags[i] = "I-" + typ
    return tags

def read(text):
    out = []
    for line in text.strip().splitlines():
        tokens, spans = parse_line(line)
        out.append((tokens, spans_to_bio(len(tokens), spans)))
    return out

train, test = read(TRAIN_TEXT), read(TEST_TEXT)
for name, data in [("oʻqitish", train), ("test", test)]:
    n_ent = Counter(t[2:] for _, tags in data for t in tags if t.startswith("B-"))
    print(f"{name}: {len(data)} gap, {sum(len(t) for t, _ in data)} token, "
          f"obyekt: {dict(n_ent)}")

tokens, tags = train[0]
print()
print(pd.DataFrame([tokens, tags], index=["token", "BIO"]).to_string(header=False))

Teskari oʻtkazish bashorat qilingan teglardan obyektlarni olish uchun kerak. Har qanday teglar ketma-ketligi ham toʻgʻri emas: I-X faqat B-X yoki I-X dan keyin kelishi mumkin. O → I-LOC yoki B-PER → I-LOC oʻtishlari — xato. Har bir token uchun tegni alohida tanlaydigan model bunday xatolarga yoʻl qoʻyishi mumkin.

In [ ]:
def bio_to_spans(tags):
    # X obyektini davom ettirmaydigan I-X yangi obyektni boshlaydi (conlleval dagi kabi)
    spans, start, typ = [], None, None
    for i, t in enumerate(list(tags) + ["O"]):
        continues = t.startswith("I-") and t[2:] == typ
        if start is not None and not continues:
            spans.append((start, i, typ))
            start, typ = None, None
        if t.startswith("B-") or (t.startswith("I-") and not continues):
            start, typ = i, t[2:]
    return spans

def bio_errors(tags):
    # ruxsat etiladi: I-X faqat B-X yoki I-X dan keyin
    errs = []
    for i, t in enumerate(tags):
        prev = tags[i - 1] if i > 0 else "O"
        if t.startswith("I-") and prev[2:] != t[2:]:
            errs.append(f"{i}: {prev} → {t}")
    return errs

examples = [
    tags,
    ["B-PER", "I-PER", "O", "O", "O", "I-LOC", "O", "O", "O", "B-ORG", "I-ORG"],
    ["B-PER", "I-LOC", "O", "O", "B-LOC", "I-LOC", "O", "O", "O", "B-ORG", "I-ORG"],
]
for ex in examples:
    errs = bio_errors(ex)
    print(" ".join(ex))
    print("   ", "toʻgʻri" if not errs else "xatolar: " + "; ".join(errs))

print("\nBIO dan tiklangan obyektlar:")
for start, end, typ in bio_to_spans(tags):
    print(f"   {typ}: {' '.join(tokens[start:end])}  [{start}, {end})")

Oʻzbek tilida obyekt tokeni koʻpincha kelishik qoʻshimchasini oladi: «Hirot**da**», «Samarqand**dagi**», «universiteti**da**». Bizning belgilashda butun token obyektga kiradi (Hirotda = B-LOC). Boshqa yoʻl — qoʻshimchani alohida token qilish: «Hirot» = B-LOC, «-da» = O. Shunda obyekt matni toza boʻladi (Hirot), lekin tokenizatsiya qoʻshimchani toʻgʻri ajratishi kerak. Oʻzbek tili uchun tayyor morfologik tahlilchilar kam, shuning uchun kelishik qoʻshimchalari roʻyxati boʻyicha oddiy qoidani oʻzimiz yozamiz.

In [ ]:
CASE_SUFFIXES = ["gacha", "dagi", "ning", "dan", "da", "ga", "ka", "qa", "ni"]

def split_case(word):
    # oxirgi kelishik qoʻshimchasini ajratamiz: Hirotda -> (hirot, da); oʻzak kamida 3 harf
    lw = word.lower()
    for suf in CASE_SUFFIXES:
        stem = lw[: -len(suf)]
        if lw.endswith(suf) and len(stem) >= 3:
            if (suf == "ka" and not stem.endswith("k")) or (suf == "qa" and not stem.endswith("q")):
                continue
            return stem, suf
    return lw, ""

for w in ["Hirotda", "Samarqanddagi", "universitetida", "Oʻzbekistonning", "Chorvoqqa", "Buxorogacha", "Shahzoda", "juda"]:
    stem, suf = split_case(w)
    print(f"   {w:<16} → {stem} + -{suf}" if suf else f"   {w:<16} → {stem}")

def split_entity_suffixes(toks, tg):
    # obyekt ichidagi oxirgi tokenning kelishik qoʻshimchasini alohida O-token qilamiz
    out_t, out_g = [], []
    for i, (w, t) in enumerate(zip(toks, tg)):
        stem, suf = split_case(w)
        last = t != "O" and (i + 1 == len(tg) or not tg[i + 1].startswith("I-"))
        if last and suf:
            out_t += [w[: len(w) - len(suf)], "-" + suf]
            out_g += [t, "O"]
        else:
            out_t.append(w)
            out_g.append(t)
    return out_t, out_g

print()
st, sg = split_entity_suffixes(tokens, tags)
print(pd.DataFrame([st, sg], index=["token", "BIO"]).to_string(header=False))

Qoida koʻp hollarda ishlaydi, lekin xato ham qiladi: «Shahzoda» ismi «Shahzo + -da» deb ajratildi, chunki qoida soʻz oʻzagini bilmaydi. «juda» ajratilmadi: oʻzak 3 harfdan qisqa boʻlib qoladi. Bunday qoidalar uchun lugʻat yoki toʻliq morfologik tahlilchi kerak (3-amaliy mashgʻulot). Bundan keyin tokenlarni boʻlmaymiz, qoʻshimchani esa belgi sifatida ishlatamiz.

## 2. Token belgilari

CRF soʻzlarni koʻrmaydi, u belgilarni koʻradi. Har bir token uchun `{belgi: qiymat}` lugʻatini tuzamiz. Satr qiymatlari indikatorlarga aylanadi (`word:hirotda`), mantiqiy qiymatlar — 0/1 ga. Belgilar guruhlari:

- **soʻz**: kichik harflardagi soʻzning oʻzi;
- **shakl**: bosh harf bilan boshlanadimi, hammasi bosh harflarmi, gapdagi birinchi soʻzmi (u yerda bosh harf hech narsani anglatmaydi);
- **kontekst**: qoʻshni soʻzlar va ularning registri;
- **morfologiya va lugʻat**: oxirgi 2–3 harf, 1-boʻlimdagi qoida boʻyicha oʻzak va kelishik, «bosh harf + kelishik» birikmasi (bosh harfli soʻz -da, -dan, -ga bilan koʻpincha joy nomi), kichik shaharlar lugʻati (gazetteer) va familiya oxirlari (-ov, -ova, -ev, -eva). Rus tilidagi `pymorphy3` kabi ismlar va joy nomlari lugʻatiga ega tayyor vosita oʻzbek tili uchun yoʻq, shuning uchun lugʻatni oʻzimiz tuzamiz.

In [ ]:
# kichik lugʻat (gazetteer): shahar nomlarining oʻzaklari
CITY_STEMS = {"toshkent", "samarqand", "buxoro", "xiva", "andijon", "fargʻona", "namangan", "qoʻqon",
              "termiz", "qarshi", "nukus", "urganch", "hirot", "navoiy"}

def lex_features(word):
    # qoʻshimchaga asoslangan belgilar: oʻzak, kelishik, «bosh harf + kelishik», shahar lugʻati, familiya oxiri
    stem, case = split_case(word)
    f = {"stem": stem, "case": case or "-", "title+case": f"{word[0].isupper()}|{case or '-'}"}
    if stem in CITY_STEMS:
        f["gaz:city"] = True
    if stem.endswith(("ov", "ova", "ev", "eva")):
        f["surname_end"] = True
    return f

ALL_GROUPS = ("word", "shape", "context", "morph")   # belgilar guruhlari: soʻz, shakl (bosh harf), kontekst (qoʻshnilar), morfologiya + lugʻat

def token_features(tokens, i, groups=ALL_GROUPS):
    w = tokens[i]
    lw = w.lower()
    f = {"bias": 1.0}
    if "word" in groups:
        f["word"] = lw
    if "shape" in groups:
        f["is_title"] = w[0].isupper()
        f["is_upper"] = w.isupper() and len(w) > 1
        f["is_first"] = i == 0
    if "context" in groups:
        f["-1:word"] = tokens[i - 1].lower() if i > 0 else "<s>"
        f["+1:word"] = tokens[i + 1].lower() if i < len(tokens) - 1 else "</s>"
        f["-1:title"] = i > 0 and tokens[i - 1][0].isupper()
        f["+1:title"] = i < len(tokens) - 1 and tokens[i + 1][0].isupper()
    if "morph" in groups:
        f["suf2"] = lw[-2:]
        f["suf3"] = lw[-3:]
        f.update(lex_features(w))
    return f

def sent_features(tokens, groups=ALL_GROUPS):
    return [token_features(tokens, i, groups) for i in range(len(tokens))]

for i in [2, 4]:
    print(f"Token belgilari «{tokens[i]}»:")
    print("   ", {k: v for k, v in token_features(tokens, i).items() if v is not False})

## 3. CRF ni oʻqitish

Chiziqli CRF (conditional random field) teglar ketma-ketligini birdaniga baholaydi:
$$\text{score}(x, y) = \sum_{i} \sum_{k} w_k\, f_k(x, i, y_i) + \sum_{i} T[y_{i-1}, y_i], \qquad P(y \mid x) = \frac{\exp \text{score}(x, y)}{Z(x)}.$$
Birinchi yigʻindi — **emissiyalar**: har bir teg uchun token belgilarining vaznlari. Ikkinchisi — **oʻtishlar**: qoʻshni teglar juftining vazni. $Z(x)$ — barcha mumkin boʻlgan belgilashlar boʻyicha $\exp \text{score}$ yigʻindisi. HMM dan farqli oʻlaroq, model soʻzlarni hosil qilmaydi, balki darhol $P(y \mid x)$ ni modellashtiradi, shuning uchun belgilar har qanday boʻlishi va bir-biri bilan kesishishi mumkin. Vaznlar oʻqitish maʼlumotlarida regulyarizatsiya bilan $\log P(y \mid x)$ ni maksimallashtirish orqali tanlanadi.

In [ ]:
import sklearn_crfsuite

X_train = [sent_features(t) for t, _ in train]
y_train = [g for _, g in train]
X_test = [sent_features(t) for t, _ in test]
y_test = [g for _, g in test]

# c1, c2 — L1 va L2 regulyarizatsiya; all_possible_transitions — barcha teglar juftlari uchun vaznlar, uchramaganlari uchun ham
crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.05, c2=0.05, max_iterations=200,
                           all_possible_transitions=True)
crf.fit(X_train, y_train)
y_pred = [list(p) for p in crf.predict(X_test)]
print(f"{len(crf.state_features_)} ta nolga teng boʻlmagan vaznli belgi\n")

print("Testdagi bashoratlar (qavs ichida — toʻgʻri teg):")
for (toks, gold), pred in zip(test, y_pred):
    print("   ", " ".join(f"{w}/{p}" + ("" if p == g else f"({g})") for w, p, g in zip(toks, pred, gold)))

Oʻrganilgan oʻtish vaznlari. Musbat vazn — teglar jufti ragʻbatlantiriladi, manfiy — jarimalanadi. B-PER → I-PER va B-ORG → I-ORG oʻtishlari kuchli musbat: koʻp soʻzli ismlar va tashkilotlar. O qatoriga qarang: O → I-PER va O → I-ORG manfiy vazn oldi. Model I- teg O dan keyin kelmasligini oʻzi oʻrgandi, biz buni unga aytmagan edik.

Quyida eng kuchli emissiya belgilari: shaharlar lugʻati (`gaz:city`), «universiteti», «vodiysi» kabi tashkilot va joy nomlarining davomi. `title+case:False|-` (kichik harfli, kelishik qoʻshimchasisiz soʻz) O uchun ovoz beradi.

In [ ]:
import matplotlib.pyplot as plt

LABELS = ["O", "B-PER", "I-PER", "B-LOC", "I-LOC", "B-ORG", "I-ORG"]
TR = np.array([[crf.transition_features_.get((a, b), 0.0) for b in LABELS] for a in LABELS])
print("Oʻtish vaznlari (qator — oldingi teg, ustun — keyingi teg):")
print(pd.DataFrame(TR, index=LABELS, columns=LABELS).round(2).to_string())

fig, ax = plt.subplots(figsize=(6.5, 5))
lim = np.abs(TR).max()
im = ax.imshow(TR, cmap="RdBu", vmin=-lim, vmax=lim)
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        ax.text(j, i, f"{TR[i, j]:.1f}", ha="center", va="center", fontsize=8)
ax.set_xticks(range(len(LABELS)), LABELS, rotation=45)
ax.set_yticks(range(len(LABELS)), LABELS)
ax.set_xlabel("keyingi teg")
ax.set_ylabel("oldingi teg")
ax.set_title("CRF oʻtish vaznlari")
plt.colorbar(im, fraction=0.04)
plt.tight_layout()
plt.show()

print("Eng kuchli emissiya belgilari:")
for (feat, label), w in Counter(crf.state_features_).most_common(12):
    print(f"   {w:6.2f}  {label:<6} {feat}")

## 4. CRF qanday qaror qabul qiladi: Viterbi va $P(y \mid x)$

Modelni tushunganimizni tekshiramiz. `crf.state_features_` va `crf.transition_features_` dan vaznlarni olamiz, har bir token emissiyalarini oʻzimiz hisoblaymiz va gapni 8-amaliy mashgʻulotdagi Viterbi algoritmi bilan dekodlaymiz. Natija `crf.predict` bilan mos kelishi kerak.

Keyin oʻtishlarni olib tashlaymiz va har bir token uchun eng yaxshi tegni alohida tanlaymiz (mustaqil klassifikator kabi). Har bir shunday gap uchun ikkala belgilashni va ularning yigʻindisini chop etamiz: emissiyalar + oʻtishlar.

In [ ]:
W = crf.state_features_

def emission_matrix(feats):
    # har bir teg uchun token barcha belgilari vaznlarining yigʻindisi (emissiyalar)
    E = np.zeros((len(feats), len(LABELS)))
    for i, f in enumerate(feats):
        for key, val in f.items():
            name, x = (key + ":" + val, 1.0) if isinstance(val, str) else (key, float(val))
            if x:
                for j, lab in enumerate(LABELS):
                    E[i, j] += W.get((name, lab), 0.0) * x
    return E

def viterbi(E, T):
    # Viterbi: 8-amaliy mashgʻulotdagi bilan bir xil, faqat ehtimollar logarifmi oʻrniga vaznlar
    n = len(E)
    delta, back = E[0].copy(), []
    for i in range(1, n):
        scores = delta[:, None] + T
        back.append(scores.argmax(axis=0))
        delta = scores.max(axis=0) + E[i]
    path = [int(delta.argmax())]
    for b in reversed(back):
        path.append(int(b[path[-1]]))
    return [LABELS[j] for j in reversed(path)]

def score(E, T, tags):
    idx = [LABELS.index(t) for t in tags]
    em = sum(E[i, j] for i, j in enumerate(idx))
    tr = sum(T[a, b] for a, b in zip(idx, idx[1:]))
    return em, tr

same, invalid = 0, []
for (toks, gold), feats, pred in zip(test, X_test, y_pred):
    E = emission_matrix(feats)
    same += viterbi(E, TR) == pred
    indep = [LABELS[j] for j in E.argmax(axis=1)]     # = viterbi(E, 0 * TR)
    if bio_errors(indep):
        invalid.append((toks, E, indep, pred))
print(f"test gaplari, ularda bizning Viterbi crf.predict bilan mos keldi: {same}/{len(test)}")
print(f"test gaplari, ularda oʻtishlarsiz notoʻgʻri BIO-belgilash chiqadi: {len(invalid)}\n")

for toks, E, indep, pred in invalid:
    print(" ".join(toks))
    for name, tg in [("Faqat emissiyalar (har bir token alohida):", indep), ("CRF (emissiyalar + oʻtishlar):", pred)]:
        em, tr = score(E, TR, tg)
        bad = "  ← notoʻgʻri BIO: " + "; ".join(bio_errors(tg)) if bio_errors(tg) else ""
        print(f"   {name:<38} {' '.join(tg)}")
        print(f"   {'':<38} emissiyalar {em:6.2f} + oʻtishlar {tr:6.2f} = yigʻindi {em + tr:6.2f}{bad}")
    print()

Oʻtishlarsiz model baʼzan I- ni O dan keyin yoki boshqa turdagi tegdan keyin qoʻyadi: «Oʻzbekiston havo yoʻllarida» da «havo» kichik harfli oddiy soʻz sifatida O oldi, «yoʻllarida» esa alohida «tashkilot qismiga oʻxshaydi». Boʻlakning yaxlitligini hech kim tekshirmaydi. CRF emissiyalarda biroz ball yoʻqotadi, lekin oʻtishlarda yutadi.

Endi ehtimol. $Z(x)$ toʻgʻri yoʻl (forward) algoritmi bilan hisoblanadi: bu Viterbi, faqat maksimum yigʻindiga almashtirilgan (logarifmlarda — `logaddexp`). Qisqa gap uchun barcha $7^n$ belgilashni saralab chiqish va $P(y \mid x)$ crfsuite oʻzi hisoblagan qiymat bilan mos kelishini, barcha belgilashlar boʻyicha yigʻindi esa 1 ga tengligini tekshirish mumkin.

In [ ]:
import itertools

def log_partition(E, T):
    # toʻgʻri yoʻl: Viterbi kabi, lekin max oʻrniga log-sum-exp; log Z(x) ni olamiz
    alpha = E[0].copy()
    for i in range(1, len(E)):
        alpha = np.logaddexp.reduce(alpha[:, None] + T, axis=0) + E[i]
    return np.logaddexp.reduce(alpha)

toks, gold = test[12]
feats = X_test[12]
E = emission_matrix(feats)
logZ = log_partition(E, TR)
tagger = crf.tagger_
tagger.set(feats)
print(" ".join(toks))
print(f"log Z(x) = {logZ:.3f}   ({len(LABELS)}^{len(toks)} = {len(LABELS) ** len(toks)} ta belgilash)\n")

cands = []
for tg in itertools.product(LABELS, repeat=len(toks)):
    em, tr = score(E, TR, tg)
    cands.append((em + tr, tg))
cands.sort(reverse=True)
print("Eng ehtimolli belgilashlar (barcha yoʻllar boʻyicha ehtimollar yigʻindisi 1 ga teng):")
for s, tg in cands[:5]:
    p_ours = np.exp(s - logZ)
    print(f"   {' '.join(tg):<34} score {s:6.2f}   P = {p_ours:.4f}   crfsuite: {tagger.probability(list(tg)):.4f}")
print(f"   Σ P = {np.exp(np.logaddexp.reduce([s for s, _ in cands]) - logZ):.4f}")

## 5. Sifat: obyektlar boʻyicha aniqlik, toʻliqlik va F1

O teglari tokenlarning katta qismini tashkil qiladi, shuning uchun tokenlar boʻyicha toʻgʻri teglar ulushi kuchsiz modelda ham yaxshi koʻrinadi. NER uchun sifat obyektlar boʻyicha hisoblanadi: obyekt faqat uning boshi, oxiri va turi mos kelsa toʻgʻri topilgan hisoblanadi.
$$P = \frac{TP}{TP + FP}, \qquad R = \frac{TP}{TP + FN}, \qquad F_1 = \frac{2PR}{P + R}.$$
Bu yerda TP — toʻgʻri topilgan obyektlar, FP — ortiqchalari (etalonda yoʻq), FN — topilmaganlari. Metrikalar haqida batafsil — 18-amaliy mashgʻulotda.

In [ ]:
def entity_prf(gold_seqs, pred_seqs):
    # qatʼiy moslik: obyektning boshi, oxiri va turi
    counts = defaultdict(lambda: [0, 0, 0])      # tp, fp, fn
    for g, p in zip(gold_seqs, pred_seqs):
        G, P = set(bio_to_spans(g)), set(bio_to_spans(p))
        for s in P:
            counts[s[2]][0 if s in G else 1] += 1
        for s in G - P:
            counts[s[2]][2] += 1
    rows = {}
    for typ in sorted(counts) + ["hammasi"]:
        tp, fp, fn = [sum(counts[t][k] for t in counts) for k in range(3)] if typ == "hammasi" else counts[typ]
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f = 2 * p * r / (p + r) if p + r else 0.0
        rows[typ] = dict(tp=tp, fp=fp, fn=fn, P=p, R=r, F1=f)
    return pd.DataFrame(rows).T

from sklearn_crfsuite import metrics
report = entity_prf(y_test, y_pred)
print(report.round(3).to_string())
print(f"\ntokenlar boʻyicha toʻgʻri teglar ulushi (O bilan birga): {metrics.flat_accuracy_score(y_test, y_pred):.3f}")
print(f"obyektlar boʻyicha F1: {report.loc['hammasi', 'F1']:.3f}")

print("\nObyektlar darajasidagi xatolar:")
for (toks, gold), pred in zip(test, y_pred):
    G, P = set(bio_to_spans(gold)), set(bio_to_spans(pred))
    for s, e, t in sorted(G - P):
        print(f"   topilmadi: {t:<4} {' '.join(toks[s:e])}")
    for s, e, t in sorted(P - G):
        print(f"   ortiqcha: {t:<4} {' '.join(toks[s:e])}")

Xatolarni tahlil qiling. Gap boshidagi «Kecha», «Ertalab» bosh harf bilan yozilgan, shuning uchun model ularni ism deb oʻyladi: birinchi soʻzdagi bosh harf ishonchsiz belgi. «Oʻzbektelekomga» joy deb topildi: bosh harf va -ga qoʻshimchasi koʻpincha joy nomida uchraydi, bu tashkilot nomi esa oʻqitish maʼlumotlarida yoʻq edi. «Kamola Orol dengizi» da ikki obyekt yonma-yon turibdi: kuchli B-PER → I-PER oʻtishi «Orol» ni ismning davomiga aylantirdi (Orol — ism ham boʻlishi mumkin).

Qaysi belgilar muhimroq? CRF ni belgilar guruhlari asta-sekin kengayib boradigan toʻplamlar bilan oʻqitamiz. 18 ta test gapida bitta obyekt F1 ning taxminan 3% iga teng, shuning uchun kichik tebranishlar — shovqin. Shakl va kontekst oʻzbek korpusida deyarli yordam bermaydi: gap boshidagi soʻz ham bosh harfli, qoʻshni soʻzlar esa kichik korpusda kam takrorlanadi. Katta sakrashni «morfologiya» guruhi beradi: qoʻshimchani ajratish bilan «Toshkentda», «Toshkentga», «Toshkentdagi» bitta «toshkent» oʻzagiga keladi, model bitta soʻz haqidagi bilimni uning barcha shakllariga qoʻllay oladi, shaharlar lugʻati esa notanish shakllarni ham qamraydi.

In [ ]:
settings = [
    ("soʻz", ("word",)),
    ("+ shakl", ("word", "shape")),
    ("+ kontekst", ("word", "shape", "context")),
    ("+ morfologiya", ALL_GROUPS),
]
f1s = []
for name, groups in settings:
    m = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.05, c2=0.05, max_iterations=200,
                             all_possible_transitions=True)
    m.fit([sent_features(t, groups) for t, _ in train], y_train)
    pred = [list(p) for p in m.predict([sent_features(t, groups) for t, _ in test])]
    rep = entity_prf(y_test, pred)
    f1s.append(rep.loc["hammasi", "F1"])
    print(f"{name:<16} P = {rep.loc['hammasi', 'P']:.3f}   R = {rep.loc['hammasi', 'R']:.3f}   F1 = {f1s[-1]:.3f}")

plt.figure(figsize=(6, 3.2))
plt.bar([n for n, _ in settings], f1s, color=["#9AA9C9", "#7088C0", "#4C6CC4", "#2B55C9"])
for i, v in enumerate(f1s):
    plt.text(i, v + 0.01, f"{v:.2f}", ha="center")
plt.ylim(0, 1.08)
plt.ylabel("F1")
plt.title("Turli belgilar toʻplamlarida testdagi obyektlar boʻyicha F1", fontsize=10)
plt.tight_layout()
plt.show()

## 6. Topshiriqlar

1. **Mustaqil klassifikator.** Xuddi shu belgilarda `LogisticRegression` ni oʻqiting (`sklearn.feature_extraction` dagi `DictVectorizer` lugʻatlarni matritsaga aylantiradi; har bir token — alohida misol). `entity_prf` funksiyasi bilan obyektlar boʻyicha F1 ni va notoʻgʻri BIO li test gaplari sonini (`bio_errors`) hisoblang. CRF bilan solishtiring.
2. **Oʻzbek belgilarining hissasi.** `lex_features` dan navbat bilan «title+case», «gaz:city» va «surname_end» ni olib tashlang va har safar CRF ni qayta oʻqiting. Qaysi belgi F1 ga koʻproq taʼsir qiladi? Qaysi xatolar paydo boʻladi?
3. **Qoʻshimchalarni ajratish.** 1-boʻlimdagi `split_entity_suffixes` funksiyasi bilan oʻqitish va test maʼlumotlarini qoʻshimchalari alohida tokenlarga ajratilgan koʻrinishga oʻtkazing. CRF ni shu maʼlumotlarda oʻqiting va F1 ni solishtiring (ikkala tokenizatsiyada obyektlar soni bir xil).
4. **Qatʼiy boʻlmagan moslik.** `entity_prf_partial` funksiyasini yozing: bashorat qilingan obyekt etalondagi xuddi shu turdagi obyekt bilan kamida bitta token boʻyicha kesishsa, toʻgʻri hisoblanadi. «morfologiya» guruhisiz model uchun qatʼiy metrika bilan solishtiring.
5. **Regulyarizatsiya.** CRF ni `c1` = 0, 0.5, 2 (`c2=0.05` da) bilan oʻqiting. Nolga teng boʻlmagan vaznli belgilar soni (`len(crf.state_features_)`) va F1 qanday oʻzgaradi? Nega L1 vaznlarni nolga aylantiradi?

*Maslahat: oʻz gaplaringizni `[matn](TUR)` formatida `my_ner.txt` ga saqlab, «Files» paneli orqali yuklashingiz va `read(open("my_ner.txt").read())` funksiyasi bilan oʻqishingiz mumkin.*

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing

In [ ]:
# Kodingizni shu yerga yozing